In [1]:
import os
import re
import pandas as pd
from pathlib import Path

# Lokasi file CSV
base_dir = Path.cwd()
file1 = base_dir / "data_belumrapih.csv"
file2 = base_dir / "data_sudah_sesuai.csv"
output_file = base_dir / "cleaned_cleanser.csv"

# Pastikan file ada
for f in [file1, file2]:
    if not f.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {f}")

# Kolom target final (urutan sesuai permintaan)
target_columns = [
    "No SPPG",
    "Nama SPPG",
    "Provinsi SPPG",
    "Kabupaten/Kota SPPG",
    "Kecamatan SPPG",
    "Latitude",
    "Longitude",
    "Alamat SPPG",
    "link",
]

# Normalisasi nama kolom agar aman dari spasi, titik, slash, dll.
def normalize_col_name(name):
    if pd.isna(name):
        return ""
    s = str(name).strip()
    s = s.replace("\ufeff", "")
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Mencari kolom berdasarkan kemungkinan variasi nama
def find_matching_column(columns, aliases):
    normalized_aliases = {normalize_col_name(a): a for a in aliases}
    for c in columns:
        n = normalize_col_name(c)
        if n in normalized_aliases:
            return c
    return None

# Mapping kolom dari berbagai kemungkinan nama menjadi nama final
col_aliases = {
    "No SPPG": [
        "no sppg", "no_sppg", "nomor sppg", "nomor_sppg", "no", "nomor"
    ],
    "Nama SPPG": [
        "nama sppg", "nama_sppg", "nama", "nama lokasi", "nama_lokasi"
    ],
    "Provinsi SPPG": [
        "provinsi sppg", "provinsi_sppg", "provinsi"
    ],
    "Kabupaten/Kota SPPG": [
        "kabupaten kota sppg", "kabupaten_kota_sppg", "kabupaten kota",
        "kabupaten/kota", "kabupaten_kota", "kabupaten", "kota"
    ],
    "Kecamatan SPPG": [
        "kecamatan sppg", "kecamatan_sppg", "kecamatan"
    ],
    "Latitude": [
        "latitude", "lat", "latitude_sppg", "lintang"
    ],
    "Longitude": [
        "longitude", "long", "lon", "lng", "longitude_sppg", "bujur"
    ],
    "Alamat SPPG": [
        "alamat sppg", "alamat_sppg", "alamat", "alamat_lengkap"
    ],
    "Link": [
        "link", "links", "url", "website", "tautan", "link_sppg"
    ],
}

def read_and_prepare_csv(path):
    # Coba baca dengan delimiter bawaan, jika gagal gunakan ;
    try:
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
    except Exception:
        df = pd.read_csv(path, sep=";", dtype=str, keep_default_na=False)

    # Normalisasi nama kolom sementara
    renamed = {}
    for col in df.columns:
        normalized = normalize_col_name(col)
        renamed[col] = normalized

    # Ubah nama kolom agar sama format
    df = df.rename(columns={orig: normalized for orig, normalized in renamed.items()})

    # Cari kolom yang cocok ke target final
    mapped_df = pd.DataFrame(index=df.index)

    for final_col, aliases in col_aliases.items():
        source_col = find_matching_column(df.columns, aliases)
        if source_col is not None:
            mapped_df[final_col] = df[source_col]

    # Jika ada kolom yang tidak cocok, tetap simpan di bawah nama aslinya
    # supaya data tidak hilang
    for col in df.columns:
        if col not in mapped_df.columns:
            mapped_df[col] = df[col]

    # Pastikan semua kolom target ada
    for col in target_columns:
        if col not in mapped_df.columns:
            mapped_df[col] = ""

    # Bersihkan whitespace
    for col in mapped_df.columns:
        if mapped_df[col].dtype == "object":
            mapped_df[col] = mapped_df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

    return mapped_df

# Baca dua file
df1 = read_and_prepare_csv(file1)
df2 = read_and_prepare_csv(file2)

# Gabungkan data (append / concat) tanpa mengubah isi data
combined = pd.concat([df1, df2], ignore_index=True, sort=False)

# Urutkan kolom sesuai permintaan
final_df = combined.copy()
for col in target_columns:
    if col not in final_df.columns:
        final_df[col] = ""

# Hanya ambil kolom sesuai urutan final
final_df = final_df[target_columns]

# Simpan hasil
final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
print(f"Berhasil dibuat: {output_file}")
print(f"Jumlah baris akhir: {len(final_df)}")

Berhasil dibuat: c:\cleansing_fix\cleaned_cleanser.csv
Jumlah baris akhir: 6234


In [ ]:
# Cek kesesuaian lokasi setiap row berdasarkan informasi pada Alamat SPPG
# Menggunakan kolom yang sudah ada: Nama SPPG, Provinsi SPPG, Kabupaten/Kota SPPG, Kecamatan SPPG, Alamat SPPG

def normalize_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower().replace("\ufeff", "")
    s = s.replace("kabupaten", "kab ").replace("kota", "kota ").replace("kecamatan", "kec ")
    s = s.replace("provinsi", "prov ").replace("prov.", "prov ").replace("kab.", "kab ").replace("kec.", "kec ")
    s = s.replace("/", " ")
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def variants(value):
    v = normalize_text(value)
    out = []
    seen = set()
    cand = [v, v.replace("kab ", " "), v.replace("kota ", " "), v.replace("kec ", " "), v.replace("prov ", " ")]
    cand += [
        v.replace("kab ", " ").replace("kota ", " "),
        v.replace("kec ", " ").replace("kab ", " "),
        v.replace("prov ", " ").replace("kota ", " "),
    ]
    for x in cand:
        x = normalize_text(x)
        if x and len(x) >= 3 and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def contains_value_in_address(address, value):
    if pd.isna(value):
        return False
    addr_norm = normalize_text(address)
    if not addr_norm:
        return False
    value_variants = variants(value)
    if not value_variants:
        return False
    return any(v in addr_norm for v in value_variants)

check_df = final_df.copy()

check_df["prov_in_alamat"] = check_df.apply(
    lambda r: contains_value_in_address(r["Alamat SPPG"], r["Provinsi SPPG"]),
    axis=1
)
check_df["kab_in_alamat"] = check_df.apply(
    lambda r: contains_value_in_address(r["Alamat SPPG"], r["Kabupaten/Kota SPPG"]),
    axis=1
)
check_df["kec_in_alamat"] = check_df.apply(
    lambda r: contains_value_in_address(r["Alamat SPPG"], r["Kecamatan SPPG"]),
    axis=1
)

check_df["lokasi_match"] = check_df["prov_in_alamat"] & check_df["kab_in_alamat"] & check_df["kec_in_alamat"]
check_df["mismatch_count"] = (
    (~check_df["prov_in_alamat"]).astype(int) +
    (~check_df["kab_in_alamat"]).astype(int) +
    (~check_df["kec_in_alamat"]).astype(int)
)

print(f"Total baris: {len(check_df)}")
print(f"Jumlah data sesuai semua (prov + kab/kota + kec): {int(check_df['lokasi_match'].sum())}")
print(f"Jumlah mismatch: {int((~check_df['lokasi_match']).sum())}")

print("\nDetail mismatch:")
print(f" - Provinsi mismatch: {int((~check_df['prov_in_alamat']).sum())}")
print(f" - Kab/Kota mismatch: {int((~check_df['kab_in_alamat']).sum())}")
print(f" - Kecamatan mismatch: {int((~check_df['kec_in_alamat']).sum())}")

mismatch_df = check_df.loc[~check_df["lokasi_match"], [
    "Nama SPPG", "Provinsi SPPG", "Kabupaten/Kota SPPG", "Kecamatan SPPG",
    "Alamat SPPG", "prov_in_alamat", "kab_in_alamat", "kec_in_alamat", "mismatch_count"
]]

if len(mismatch_df) > 0:
    print("\nContoh data mismatch:")
    print(mismatch_df.head(20).to_string(index=False))
else:
    print("\nTidak ada data mismatch pada pengecekan berdasarkan Alamat SPPG.")

mismatch_df

Total baris: 6234
Jumlah data sesuai semua (prov + kab/kota + kec): 5204
Jumlah mismatch: 1030

Detail mismatch:
 - Provinsi mismatch: 974
 - Kab/Kota mismatch: 110
 - Kecamatan mismatch: 23

Contoh data mismatch:
                                      Nama SPPG    Provinsi SPPG Kabupaten/Kota SPPG     Kecamatan SPPG                                                                                                                                                Alamat SPPG  prov_in_alamat  kab_in_alamat  kec_in_alamat  mismatch_count
               SPPG ACEH BARAT MEUREUBO LANGUNG             Aceh          Aceh Barat           Meureubo                                                                JL. Nasional Meulaboh No.1105, Tapak Tuan, Langung, Meureubo, West Aceh Regency, Aceh 23615            True          False           True               1
                            SPPG KRUENG SEUNONG             Aceh          Aceh Utara        Kuta Makmur                                         

,Nama SPPG,Provinsi SPPG,Kabupaten/Kota SPPG,Kecamatan SPPG,Alamat SPPG,prov_in_alamat,kab_in_alamat,kec_in_alamat,mismatch_count
0,SPPG ACEH BARAT MEUREUBO LANGUNG,Aceh,Aceh Barat,Meureubo,"JL. Nasional Meulaboh No.1105, Tapak Tuan, Lan...",True,False,True,1
6,SPPG KRUENG SEUNONG,Aceh,Aceh Utara,Kuta Makmur,"Jl. Elak No.KM. 8, Krueng Seunong, Kuta Makmur...",True,False,True,1
7,SPPG GUMPANG JAYA 02 BABUSSALAM (YMKJ),Aceh,Aceh Tenggara,Babussalam,"Jl. Terminal Terpadu Gang Keluarga, Gumpang Ja...",True,False,True,1
8,SPPG KAMPUNG BARU,Aceh,Aceh Tenggara,Badar,"GQFP+RH5 SPPG Kampung Baru, Kampung Baru, Bada...",True,False,True,1
9,SPPG SINGGAH MATA 02,Aceh,Aceh Utara,Lhoksukon,"4964+4WM SPPG Singgah Mata 02, Singgah Mata, K...",True,False,True,1
...,...,...,...,...,...,...,...,...,...
1082,SPPG YAYASAN TARE KARSA NUSANTARA SATUAN PELAY...,Sulawesi Tengah,Kota Palu,Tatanga,3VF4+2M9 SPPG Yayasan Tare Karsa Nusantara Sat...,False,True,True,1
1083,SPPG BUMIRAYA,Sulawesi Tengah,Morowali,Bumi Raya,"QM8F+Q6X SPPG BUMIRAYA, Pebatae, Bumi Raya, Mo...",False,True,True,1
1084,SPPG POLRI POLRES KONUT,Sulawesi Tenggara,Konawe Utara,Wiwirano,"P575+RX3 SPPG POLRI POLRES KONUT, Hialu Utama ...",False,False,True,2
1086,SPPG SIGI GUMBASA PAKULI,Sulawesi Tengah,Sigi,Gumbasa,"QW6V+63G SPPG Sigi Gumbasa Pakuli, Jl. Poros P...",False,True,True,1
